# Final Evaluation on the Untouched 9 Images

Run this notebook **after** you finish parameter tuning and manually copy the selected defaults into `banana_quality/config.py`.

This notebook:

- recreates the same fixed random 21/9 split from the full pool of 30 images,
- does **not** split images by `banana`, `freshunripe`, or `mold`,
- evaluates only the **9 final-evaluation images**,
- reads the selected defaults directly from `ProcessingConfig`,
- reports banana IoU/Dice, blemish IoU/Dice, and blemish-percentage MAE,
- does not tune or change any parameter.

For a clean final evaluation, do not use these 9 images to choose new parameter values afterward.

In [1]:
from pathlib import Path
import random
import sys

import cv2
import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "banana_quality" / "config.py").exists() and (candidate / "data" / "annotations").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root. Start Jupyter from inside this project.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from banana_quality.config import ProcessingConfig
from banana_quality.pipeline import analyse_image
from banana_quality.surface_quality import blemish_percentage

DATASET_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_DIR = PROJECT_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\enton\Project\python\banana_quality_ripeness_assessment_system


In [2]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
SPLIT_SEED = 42
FINAL_EVALUATION_COUNT = 9

images_dir = DATASET_ROOT / "images"
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

# Recreate the same random split used by 01_parameter_tuning.ipynb.
shuffled_paths = image_paths.copy()
rng = random.Random(SPLIT_SEED)
rng.shuffle(shuffled_paths)
final_names = {p.name for p in shuffled_paths[:FINAL_EVALUATION_COUNT]}

print(f"Total images: {len(image_paths)}")
print(f"Tuning images: {len(image_paths) - len(final_names)}")
print(f"Final-evaluation images: {len(final_names)}")

assert len(image_paths) == 30, f"Expected 30 images, found {len(image_paths)}"
assert len(final_names) == 9

Total images: 30
Tuning images: 21
Final-evaluation images: 9


## Configuration being evaluated

These values come from `banana_quality/config.py`. If you edited the file while this notebook kernel was already running, restart the kernel before continuing.

In [3]:
config = ProcessingConfig()

parameters_used = {
    "kmeans_k": config.kmeans_k,
    "banana_cluster_score_threshold": config.banana_cluster_score_threshold,
    "banana_opening_kernel": config.banana_opening_kernel,
    "banana_closing_kernel": config.banana_closing_kernel,
    "blemish_kmeans_k": config.blemish_kmeans_k,
    "blemish_cluster_chroma_threshold": config.blemish_cluster_chroma_threshold,
    "darkness_threshold": config.darkness_threshold,
    "blemish_opening_kernel": config.blemish_opening_kernel,
    "blemish_closing_kernel": config.blemish_closing_kernel,
}

pd.DataFrame([parameters_used])

,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,blemish_kmeans_k,blemish_cluster_chroma_threshold,darkness_threshold,blemish_opening_kernel,blemish_closing_kernel
0,7,0.35,3,7,6,21.0,32.0,3,3


In [4]:
def matching_mask(folder: Path, image_path: Path) -> Path:
    path = folder / f"{image_path.stem}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_binary_mask(path: Path, target_shape: tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {path}")
    mask = np.where(mask > 0, 255, 0).astype(np.uint8)
    if mask.shape != target_shape:
        h, w = target_shape
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask


def iou_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    union = int(np.logical_or(pred, gt).sum())
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, gt).sum() / union)


def dice_score(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    pred = prediction > 0
    gt = ground_truth > 0
    denominator = int(pred.sum() + gt.sum())
    if denominator == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denominator)

## Run the final evaluation

In [5]:
rows = []

for image_path in image_paths:
    if image_path.name not in final_names:
        continue

    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")

    shape = image.shape[:2]
    banana_gt = load_binary_mask(
        matching_mask(DATASET_ROOT / "banana_masks", image_path), shape
    )
    blemish_gt = load_binary_mask(
        matching_mask(DATASET_ROOT / "blemish_masks", image_path), shape
    )

    result = analyse_image(
        image,
        segmentation_mode="mode_a",
        kmeans_k=int(config.kmeans_k),
        banana_opening_kernel=int(config.banana_opening_kernel),
        banana_closing_kernel=int(config.banana_closing_kernel),
        blemish_opening_kernel=int(config.blemish_opening_kernel),
        blemish_closing_kernel=int(config.blemish_closing_kernel),
        config=config,
    )

    banana_pred = np.asarray(result["banana_mask"], dtype=np.uint8)
    blemish_pred = np.asarray(result["blemish_mask"], dtype=np.uint8)

    gt_pct = blemish_percentage(banana_gt, blemish_gt)
    pred_pct = float(result["blemish_percentage"])

    rows.append({
        "image": image_path.name,
        "banana_iou": iou_score(banana_pred, banana_gt),
        "banana_dice": dice_score(banana_pred, banana_gt),
        "blemish_iou": iou_score(blemish_pred, blemish_gt),
        "blemish_dice": dice_score(blemish_pred, blemish_gt),
        "ground_truth_blemish_pct": gt_pct,
        "predicted_blemish_pct": pred_pct,
        "blemish_percentage_error": abs(pred_pct - gt_pct),
    })

final_results = pd.DataFrame(rows).sort_values("image").reset_index(drop=True)
final_results.to_csv(RESULTS_DIR / "final_evaluation_per_image.csv", index=False)
final_results

,image,banana_iou,banana_dice,blemish_iou,blemish_dice,ground_truth_blemish_pct,predicted_blemish_pct,blemish_percentage_error
0,musa-acuminata-banana-8e3c133d-394a-11ec-bc37-...,0.885436,0.939237,0.469477,0.638971,40.541400,36.100109,4.441291
1,musa-acuminata-banana-8f7de794-394a-11ec-a8cc-...,0.864546,0.927353,0.440224,0.611327,39.844137,23.602658,16.241479
2,musa-acuminata-banana-afe6ec6d-394a-11ec-bcab-...,0.976443,0.988081,0.005991,0.011910,3.907305,0.022918,3.884387
3,musa-acuminata-banana-b01b3700-394a-11ec-b4a6-...,0.975709,0.987705,0.297297,0.458333,5.732035,1.685068,4.046968
4,musa-acuminata-banana-b01d2f7c-394a-11ec-a466-...,0.974754,0.987216,0.343779,0.511660,8.439791,3.562369,4.877422
5,musa-acuminata-banana-b2a5cf68-394a-11ec-b7c7-...,0.988356,0.994144,0.333509,0.500198,8.059516,3.124403,4.935114
6,musa-acuminata-freshunripe-1da23724-2653-11ec-...,0.902149,0.948558,0.655111,0.791622,2.812789,2.291341,0.521448
7,musa-acuminata-freshunripe-1fd8dd9a-2653-11ec-...,0.970623,0.985092,0.393438,0.564701,2.666086,6.057100,3.391014
8,musa-acuminata-freshunripe-2cdf79be-2653-11ec-...,0.842397,0.914458,0.000000,0.000000,0.000000,13.711450,13.711450


In [6]:
summary = pd.DataFrame([{
    "images": len(final_results),
    **parameters_used,
    "banana_mean_iou": float(final_results["banana_iou"].mean()),
    "banana_mean_dice": float(final_results["banana_dice"].mean()),
    "blemish_mean_iou": float(final_results["blemish_iou"].mean()),
    "blemish_mean_dice": float(final_results["blemish_dice"].mean()),
    "blemish_percentage_mae": float(final_results["blemish_percentage_error"].mean()),
}])

summary.to_csv(RESULTS_DIR / "final_evaluation_summary.csv", index=False)
summary

,images,kmeans_k,banana_cluster_score_threshold,banana_opening_kernel,banana_closing_kernel,blemish_kmeans_k,blemish_cluster_chroma_threshold,darkness_threshold,blemish_opening_kernel,blemish_closing_kernel,banana_mean_iou,banana_mean_dice,blemish_mean_iou,blemish_mean_dice,blemish_percentage_mae
0,9,7,0.35,3,7,6,21.0,32.0,3,3,0.931157,0.963538,0.326536,0.454303,6.227841


## How to report it

Use the final summary above as the performance of the **fixed automatic Mode A configuration**. The Streamlit sliders can still allow optional user refinement during interactive use, but those per-image adjustments should not be mixed into the reported automatic IoU/Dice evaluation.